# COMPACT-MI-RFE CICFLOW — Colab V2

Bản CICFLOW theo cách tổ chức và viết code của `COMPACT_MI_RFE_ColabPro.ipynb`: mỗi cell có mục đích, điều kiện đầu vào, từng bước thực thi và kết quả/đường dẫn báo cáo rõ ràng. Cell metadata ghi chính xác cell nguồn dùng làm tham chiếu.

Giữ 15 cell theo yêu cầu CICFLOW: tách MI và RFE, kiểm tra từng stage riêng. Source đọc từ `MyDrive/code_CICFLOW`, Train từ `MyDrive/CICFLOWMETER_FEATURE_CSV`, kết quả ghi vào fresh experiment trên Drive. Không sử dụng workflow CSV WINDOWS_CPLUS, classifier final/Test hoặc SVM probe của notebook tham chiếu.

Mặc định `MODE="dry_run"`, xác nhận để trống. Chạy Cell 1–15 theo thứ tự; chỉ Cell 10 và 12 có thể fit Train khi người dùng chọn full/resume và các gate đạt yêu cầu. Scientific status giữ `PROVISIONAL_RESEARCH`.


## Cell 1 — Runtime information và cấu hình

Cấu hình chung, không tạo Run ID và không fit. Giữ hai cờ xác nhận False trong dry-run. GPU không được giả định tăng tốc sklearn. Không tự cài/hard-pin dependency.


In [ ]:
# CELL 1 — Runtime information and CICFLOW configuration
# Reference: stage6-cell-1. Read-only runtime inspection; CPU sklearn paths.
from pathlib import Path
import os, platform, shutil, sys, importlib, importlib.util
import importlib.metadata as im
import json

# ============================================================
# 1. EXECUTION MODE — safe default, no automatic fit
# ============================================================
MODE = "dry_run"  #@param ["dry_run", "full", "resume", "verify"]
EXISTING_RESULT_ROOT = ""  #@param {type:"string"}
CONFIRM_PROVISIONAL_RESEARCH = False  #@param {type:"boolean"}
REFERENCE_RECONCILIATION_CONFIRMED = False  #@param {type:"boolean"}
CONFIRMATION = ""  #@param {type:"string"}
RUN_UNIT_TESTS = True  #@param {type:"boolean"}
GIT_COMMIT = "NOT_CAPTURED"  # user-supplied identifier; no Git command is run

# ============================================================
# 2. ACTUAL GOOGLE DRIVE LAYOUT
# ============================================================
DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_ROOT = DRIVE_ROOT
SOURCE_ROOT = DRIVE_ROOT / "code_CICFLOW"
DATASET_ROOT = DRIVE_ROOT / "CICFLOWMETER_FEATURE_CSV"
PIPELINE_FILE = SOURCE_ROOT / "colab/colab_pipeline.py"
V2_PIPELINE_FILE = SOURCE_ROOT / "colab/colab_pipeline_v2.py"
OUTPUT_BASE = DRIVE_ROOT / "COMPACT_MI_RFE_FRESH_RUNS"
CONFIGS = ("FW256_FS8", "FW256_FS16", "FW512_FS8", "FW512_FS16")
BRANCHES = ("rf", "svm")
INPUT_EVIDENCE_FILES = {}  # optional existing JSON integrity/schema reports

# ============================================================
# 3. INITIALIZE NOTEBOOK STATE
# ============================================================
DEPENDENCIES = {"status": "NOT_RUN"}
UNIT_TEST_REPORT = {"status": "NOT_RUN"}
UNIT_TEST_EVIDENCE = {"passed": False}
DATA_AUDIT = {"status": "NOT_RUN", "configs": {}}
DATASET_READY = False
PREFLIGHT = {"status": "NOT_RUN"}
MI_RESULTS, MI_VERIFY, RFE_RESULTS, RFE_VERIFY = [], [], [], []
if MODE not in {"dry_run", "full", "resume", "verify"}:
    raise ValueError("Unsupported MODE")

# ============================================================
# 4. RUNTIME INFORMATION — same purpose as reference Cell 1
# ============================================================
try:
    import psutil
    vm = psutil.virtual_memory()
    ram = {"total": vm.total, "available": vm.available}
    physical = psutil.cpu_count(logical=False)
except ImportError:
    ram, physical = {"total": None, "available": None}, None
versions = {}
for name in ("numpy", "pandas", "scipy", "scikit-learn", "joblib", "pyarrow", "psutil"):
    try:
        versions[name] = im.version(name)
    except im.PackageNotFoundError:
        versions[name] = None
print({"os": platform.platform(), "arch": platform.machine(), "python": sys.version,
       "logical_cpus": os.cpu_count(), "physical_cpus": physical, "ram_bytes": ram,
       "disk_free_bytes": shutil.disk_usage("/content").free,
       "gpu_probe_available": bool(shutil.which("nvidia-smi")), "versions": versions})
print("This scikit-learn MI/RFE pipeline runs on CPU; High-RAM is not assumed.")
print("MODE:", MODE, "| scientific status: PROVISIONAL_RESEARCH")


## Cell 2 — Mount Drive và xác minh đường dẫn

Drive chứa `code_CICFLOW` và `CICFLOWMETER_FEATURE_CSV` ngay dưới MyDrive. Không thêm lớp thư mục COMPACT-MI-RFE. Cell này chưa khởi tạo thí nghiệm.


In [ ]:
# CELL 2 — Mount Drive, then resolve mounted dataset/source folders
# Reference: stage6-cell-2 and stage6-cell-3. No dataset transfer.
from google.colab import drive
if not DRIVE_ROOT.is_dir():
    drive.mount("/content/drive")
print("Drive mounted:", DRIVE_ROOT.is_dir())

print("\n=== GOOGLE DRIVE PATH CHECK ===")
for name, path in (
    ("DRIVE_ROOT", DRIVE_ROOT), ("PROJECT_ROOT", PROJECT_ROOT),
    ("SOURCE_ROOT", SOURCE_ROOT), ("DATASET_ROOT", DATASET_ROOT),
):
    print(name, "FOUND" if path.is_dir() else "MISSING")
    print(" ", path)
    if not path.is_dir():
        raise FileNotFoundError(f"{name} not found: {path}")

print("\n=== REQUIRED SOURCE CHECK ===")
required = (
    "colab/colab_pipeline.py", "colab/colab_pipeline_v2.py",
    "stage4_mi/mi_selection.py", "stage5_rfe/rfe_selection.py",
)
missing = [relative for relative in required if not (SOURCE_ROOT / relative).is_file()]
for relative in required:
    print("PASS" if relative not in missing else "MISSING", SOURCE_ROOT / relative)
if missing:
    raise FileNotFoundError(f"Upload the selected CICFLOW source first: {missing}")
print("No dataset copied, downloaded, extracted or moved.")


## Cell 3 — Import source chính xác và khởi tạo Run ID có kiểm soát

Import V2 bằng đường dẫn file, thay namespace `colab` xung đột, kiểm tra `__file__`. Fresh run được tạo đúng một lần trong session; resume/verify phải đọc V2 RUN_INFO của experiment đã chọn. Source hash được khóa, không coi bản Drive giống bản Windows khi chưa so sánh.


In [ ]:
# CELL 3 — Bind the exact Drive source and initialize/read the experiment
# Reference: stage6-cell-4 source inventory + Cell 4A cache invalidation.
# CICFLOW source is already on Drive: no clone, pull, reset or source patch.
sys.path[:] = [str(SOURCE_ROOT)] + [p for p in sys.path if p != str(SOURCE_ROOT)]
spec = importlib.util.spec_from_file_location("_cicflow_drive_v2", V2_PIPELINE_FILE)
V2 = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = V2
spec.loader.exec_module(V2)
if Path(V2.__file__).resolve() != V2_PIPELINE_FILE.resolve():
    raise RuntimeError(f"Wrong V2 source: {V2.__file__}")
PIPELINE = V2.activate_source(SOURCE_ROOT)
if Path(PIPELINE.__file__).resolve() != PIPELINE_FILE.resolve():
    raise RuntimeError(f"Wrong pipeline import: {PIPELINE.__file__}")
print("Expected:", PIPELINE_FILE)
print("Loaded:  ", PIPELINE.__file__)

previous_session_root = globals().get("ACTIVE_RESULT_ROOT", "")
initialized = V2.initialize_run(
    DRIVE_ROOT, SOURCE_ROOT, DATASET_ROOT, MODE,
    existing=EXISTING_RESULT_ROOT,
    session_root=previous_session_root if MODE in {"dry_run", "full"} else "",
)
RESULT_ROOT = Path(initialized["root"])
ACTIVE_RESULT_ROOT = str(RESULT_ROOT)
RUN_INFO = initialized["info"]
RUN_IDS = RUN_INFO["run_ids"]
SOURCE_SNAPSHOT = V2.source_snapshot(SOURCE_ROOT)
binding_path = RESULT_ROOT / "SOURCE_BINDING.json"
if binding_path.exists():
    if V2.read_json(binding_path) != SOURCE_SNAPSHOT:
        raise RuntimeError("Source differs from the saved experiment; preserve it and use its bound code or a new run")
else:
    if MODE in {"resume", "verify"}:
        raise RuntimeError("UNSUPPORTED: selected experiment has no V2 source binding")
    V2.write_json_once(binding_path, SOURCE_SNAPSHOT)

BASE = {
    "source": str(SOURCE_ROOT), "dataset": str(DATASET_ROOT), "root": str(RESULT_ROOT),
    "mode": MODE, "git_commit": GIT_COMMIT, "evidence_files": INPUT_EVIDENCE_FILES,
    "configs": list(CONFIGS), "branches": list(BRANCHES),
}
print("\n=== SOURCE IDENTITY ===")
print("SHA256:", SOURCE_SNAPSHOT["sha256"])
print("\n=== EXPERIMENT PATHS ===")
print("EXPERIMENT ID:", RUN_INFO["experiment_id"])
print("RUN IDS:", json.dumps(RUN_IDS, indent=2))
print("RESULT_ROOT:", RESULT_ROOT)
print("Stage 4:", RESULT_ROOT / "stage4_mi/runs")
print("Stage 5:", RESULT_ROOT / "stage5_rfe/runs")
print("Required full/resume confirmation:", "RUN PROVISIONAL " + RESULT_ROOT.name)
print("No MI/RFE training executed.")


## Cell 4 — Kiểm tra cấu trúc bốn dataset

Chỉ xác nhận directory và đúng một tên Train Parquet được hỗ trợ trong mỗi cấu hình. Validation/Test không được mở hoặc copy.


In [ ]:
# CELL 4 — Inspect the four prepared sliding-window dataset folders
# Reference: dataset inventory in stage6-cell-3/6; CICFLOW uses Train Parquet.
print("=== CICFLOW DATASET STRUCTURE ===")
STRUCTURE = V2.dataset_structure(DATASET_ROOT)
structure_report = V2.save_report(RESULT_ROOT, "dataset_structure", STRUCTURE)
for config in CONFIGS:
    row = STRUCTURE["configs"][config]
    print("\n", config)
    print("Path:", row["directory"])
    print("Status:", row["status"])
    print("Train file:", row["train_candidates"])
    if row["reason"]:
        print("Reason:", row["reason"])
print("\nReport:", structure_report)
if STRUCTURE["status"] != "PASS":
    raise RuntimeError("Dataset inventory failed; inspect each configuration above")
print("All four CICFLOW configurations found with one supported Train file each.")
print("No Validation/Test feature values were opened.")


## Cell 5 — Dependencies và Unit Tests bằng fresh Python

Không hard-pin NumPy/sklearn cũ trên Python 3.13. Worker kiểm tra import và lưu versions/resolved estimator defaults. Test loader tìm `test_*.py` thật, không yêu cầu thư mục tests là package. PASS, FAIL, SKIPPED, NO_TESTS_FOUND được phân biệt; mỗi area phải có test PASS trước full/resume.


In [ ]:
# CELL 5 — Dependencies, fresh import test, source tests, evidence
# Reference: stage6-cell-5. No unverified old package pins or automatic pip install.
print("=== ACTUAL PYTHON / SOURCE ===")
print("Python:", sys.version)
print("Source:", SOURCE_ROOT)

# ============================================================
# 1. SCIENTIFIC IMPORTS IN A FRESH PYTHON PROCESS
# ============================================================
dep_execution = V2.run_worker("dependencies", BASE)
DEPENDENCIES = dep_execution["result"]
print("\n=== SCIENTIFIC VERSION CHECK ===")
for name, version in DEPENDENCIES.get("versions", {}).items():
    print(f"{name}: {version}")
for error in DEPENDENCIES.get("errors", []):
    print("IMPORT ERROR:", error)
print("Import worker exit code:", dep_execution["exit_code"])
print("Dependency report:", dep_execution["report_path"])
if DEPENDENCIES.get("status") != "PASS":
    raise RuntimeError("Scientific import test failed; review the report before installing/changing packages")

# ============================================================
# 2. ACTUAL-FILE UNIT TESTS IN A FRESH PYTHON PROCESS
# ============================================================
if RUN_UNIT_TESTS:
    test_execution = V2.run_worker("tests", BASE)
    UNIT_TEST_REPORT = test_execution["result"]
    print("\n=== SOURCE AND ADAPTER UNIT TESTS ===")
    for area, result in UNIT_TEST_REPORT.get("areas", {}).items():
        print(area, result["status"], "| tests:", result["tests_run"],
              "| failed:", result["failures"], "| errors:", result["errors"],
              "| skipped:", result["skipped"])
        print("Files:", result["files"])
    print("Unit test report:", test_execution["report_path"])
else:
    UNIT_TEST_REPORT = {"status": "SKIPPED", "reason": "RUN_UNIT_TESTS=False"}
    V2.save_report(RESULT_ROOT, "unit_tests_skipped", UNIT_TEST_REPORT)

# ============================================================
# 3. SAVE EVIDENCE BOUND TO THE TESTED SOURCE
# ============================================================
UNIT_TEST_EVIDENCE = {
    "passed": UNIT_TEST_REPORT.get("status") == "PASS",
    "source_sha256": UNIT_TEST_REPORT.get("source_snapshot", {}).get("sha256"),
    "report_status": UNIT_TEST_REPORT.get("status", "UNKNOWN"),
}
V2.save_report(RESULT_ROOT, "unit_test_evidence", UNIT_TEST_EVIDENCE)
print("\nCELL 5:", UNIT_TEST_REPORT.get("status"))
print("UNIT_TEST_EVIDENCE:", UNIT_TEST_EVIDENCE)
if not UNIT_TEST_EVIDENCE["passed"]:
    print("Full/resume is BLOCKED. Dry-run diagnostics may continue.")
print("Python 3.13 compatibility is determined by the import/test results above; no bitwise reference parity is claimed.")


## Cell 6 — Metadata audit

Schema, 380 feature + Window_Label, numeric types, Train row count, schema/evidence hashes được đọc từ Train footer/JSON. Mỗi cấu hình luôn có kết quả hoặc lý do BLOCKED; không có `DATA_AUDIT={}` im lặng. Fingerprint hiện tại không phải SHA256 toàn bộ Train; độc lập capture/session và leakage vẫn chưa được chứng minh.


In [ ]:
# CELL 6 — Validate Train metadata in a clean Python process
# Reference: stage6-cell-7 fresh-worker audit and structured JSON result.
if DEPENDENCIES.get("status") != "PASS":
    raise RuntimeError("Run Cell 5 and resolve scientific import failures first")
if STRUCTURE.get("status") != "PASS":
    raise RuntimeError("Run Cell 4 and resolve the dataset structure first")

print("=== CICFLOW DATASET AUDIT ===")
print("Dataset:", DATASET_ROOT)
print("Running validation in an isolated Python process...")
audit_execution = V2.run_worker("metadata", BASE)
DATA_AUDIT = audit_execution["result"]
counts = DATA_AUDIT.get("configs", {})
DATASET_READY = (
    DATA_AUDIT.get("status") == "PASS"
    and set(counts) == set(CONFIGS)
    and all(counts[config].get("status") == "PASS" for config in CONFIGS)
)

print("\n=== VALIDATION RESULTS ===")
print("Status:", DATA_AUDIT.get("status", "NO_AUDIT_RETURNED"))
print("Configurations returned:", len(counts))
for config in CONFIGS:
    row = counts.get(config, {"status": "BLOCKED", "errors": ["No result returned"]})
    print("\n", config, row["status"])
    print("Train windows:", row.get("train_footer", {}).get("rows"))
    print("Feature count:", len(row.get("feature_names", [])))
    print("Errors:", row.get("errors", []))
    print("Split/provenance:", row.get("split_provenance_status", "NOT_VERIFIED"))
print("\nDATASET_READY:", DATASET_READY)
print("Worker exit code:", audit_execution["exit_code"])
print("Audit report:", audit_execution["report_path"])
print("Train values are not read during this metadata audit.")
print("Validation/Test feature values are not opened.")
if not DATASET_READY:
    print("Dataset audit did not pass. Full/resume remains BLOCKED.")


## Cell 7 — Scientific readiness

Giữ thuật toán source hiện tại: MI gọi từng feature với seed 42+index, không external StandardScaler; hai nhánh RFE đều dùng RandomForestClassifier với depth khác nhau. Không gọi nhánh svm là SVM-RBF RFE. Hai cờ xác nhận mặc định False; reference flag không tạo bằng chứng official. Provisional research được phép khi người dùng xác nhận rõ và các technical gate PASS.


In [ ]:
# CELL 7 — Scientific readiness and actual source parameters
# Reference: readiness gates preceding experiment launch; no fit in this cell.
SCIENTIFIC_READINESS = {
    "status": "PROVISIONAL_RESEARCH",
    "reference_status": "PROVISIONAL_REFERENCE_UNVERIFIED",
    "official_reproduction": False,
    "reference_confirmation_recorded": REFERENCE_RECONCILIATION_CONFIRMED,
    "mi_parameters": DEPENDENCIES.get("mi_parameters"),
    "rfe_contracts": DEPENDENCIES.get("rfe_contracts"),
    "resolved_rfe_estimators": DEPENDENCIES.get("resolved_rfe_estimators"),
    "metadata_status": DATA_AUDIT.get("status"),
    "unit_test_status": UNIT_TEST_REPORT.get("status"),
    "dataset_ready": DATASET_READY,
    "validation_test_values_read": False,
    "capture_session_independence": "NOT_VERIFIED",
    "xgboost": "BLOCKED",
}
readiness_report = V2.save_report(RESULT_ROOT, "scientific_readiness", SCIENTIFIC_READINESS)
print(json.dumps(SCIENTIFIC_READINESS, indent=2))
print("Readiness report:", readiness_report)
print("The svm RFE branch uses the source RandomForest estimator, not an SVM-RBF estimator.")
print("MI/RFE source parameters have not been replaced by the reference notebook's classifier/probe workflow.")


## Cell 8 — Chọn configurations và branches

Theo Cell 8 bản tham chiếu: chọn EXPERIMENT, FW, FS qua Colab form, tạo RunSpec và validate. CONFIG_SCOPE=all xử lý cả bốn cấu hình; single dùng FW/FS đã chọn. EXPERIMENT=both chạy hai nhánh RF/SVM. Run IDs được đọc từ RUN_INFO; Cell 9 phải chạy lại sau mỗi thay đổi lựa chọn.


In [ ]:
# CELL 8 — Select experiment/configuration; default mode remains DRY RUN
# Reference: stage6-cell-8 forms + RunSpec.validate().
EXPERIMENT = "both"  #@param ["both", "mi_rfe_rf", "mi_rfe_svm"]
CONFIG_SCOPE = "all"  #@param ["all", "single"]
FW = "256"  #@param ["256", "512"]
FS = 8  #@param [8, 16]

if EXPERIMENT not in {"both", "mi_rfe_rf", "mi_rfe_svm"} or CONFIG_SCOPE not in {"all", "single"}:
    raise ValueError("Unknown experiment or configuration scope")
SELECTED_CONFIGS = CONFIGS if CONFIG_SCOPE == "all" else (f"FW{FW}_FS{FS}",)
SELECTED_BRANCHES = BRANCHES if EXPERIMENT == "both" else (
    "rf" if EXPERIMENT == "mi_rfe_rf" else "svm",
)
V2.ensure_selection(SELECTED_CONFIGS, SELECTED_BRANCHES)
SPECS = [
    V2.RunSpec("mi_rfe_rf" if branch == "rf" else "mi_rfe_svm", config, RUN_IDS[config])
    for config in SELECTED_CONFIGS for branch in SELECTED_BRANCHES
]
for SPEC in SPECS:
    SPEC.validate()
    print({"experiment": SPEC.experiment, "config": SPEC.config,
           "branch": SPEC.branch, "run": SPEC.run_name, "mode": MODE})
BASE.update({"configs": list(SELECTED_CONFIGS), "branches": list(SELECTED_BRANCHES)})
# Selection changes invalidate the notebook's previous preflight.
PREFLIGHT = {"status": "NOT_RUN", "reason": "Run Cell 9 for this selection"}
print("Output:", RESULT_ROOT)
print("Full/resume requires a successful Cell 9 and the exact confirmation phrase.")


## Cell 9 — Preflight và khóa identity

Full/resume cần Unit Tests PASS, source đúng hash, metadata hợp lệ, RAM/disk, quyền ghi, output không xung đột và xác nhận bằng câu chính xác. Resource gate giữ policy của source hiện có. Reference flag được lưu riêng. Để chạy provisional, người dùng tự điền hai dòng xác nhận bên dưới; không cần khai sai rằng reference đã được chứng minh.


In [ ]:
# CELL 9 — SAFE PREFLIGHT IN A FRESH PYTHON PROCESS
# Reference: stage6-cell-9 prerequisite checks, payload, checks/RAM output.
# To authorize provisional full/resume, edit Cell 1 or set these explicitly here:
# CONFIRM_PROVISIONAL_RESEARCH = True
# CONFIRMATION = "RUN PROVISIONAL <exact experiment_id from Cell 3>"

print("=== CICFLOW PREFLIGHT ===")
print("Python:", sys.version.split()[0])
print("Dataset:", DATASET_ROOT)
print("Configurations:", SELECTED_CONFIGS)
print("Branches:", SELECTED_BRANCHES)
print("Experiment:", RUN_INFO["experiment_id"])
print("Mode:", MODE)
print("DATASET_READY:", DATASET_READY)
print("UNIT_TEST_EVIDENCE:", UNIT_TEST_EVIDENCE)
for SPEC in SPECS:
    SPEC.validate()

# ============================================================
# 1. PASS THE CURRENT CONFIGURATION TO THE CLEAN PROCESS
# ============================================================
BASE.update({
    "mode": MODE, "test_report": UNIT_TEST_REPORT,
    "confirm_provisional": CONFIRM_PROVISIONAL_RESEARCH,
    "reference_confirmed": REFERENCE_RECONCILIATION_CONFIRMED,
    "confirmation": CONFIRMATION, "git_commit": GIT_COMMIT,
})
preflight_execution = V2.run_worker("preflight", BASE)
PREFLIGHT = preflight_execution["result"]

# ============================================================
# 2. SHOW CHECKS, RAM/DISK AND DIAGNOSTICS
# ============================================================
print("\n=== PREFLIGHT RESULTS ===")
print("PREFLIGHT:", PREFLIGHT.get("status", "NO_RESULT"))
for name, passed in PREFLIGHT.get("checks", {}).items():
    print(" ", name, "PASS" if passed else "FAIL")
for config, resource in PREFLIGHT.get("resources", {}).items():
    print("\nRAM/DISK GUARD:", config)
    print("MI:", json.dumps(resource["mi"], indent=2))
    print("RFE:", json.dumps(resource["rfe"], indent=2))
print("\nErrors:", PREFLIGHT.get("errors", []))
print("Worker exit code:", preflight_execution["exit_code"])
print("Report:", preflight_execution["report_path"])
print("Preflight requests no MI/RFE fit and opens no Validation/Test feature values.")

# ============================================================
# 3. ENFORCE THE GATE
# ============================================================
if PREFLIGHT.get("status") != "PASS":
    print("Full training is BLOCKED.")
    if MODE in {"full", "resume"}:
        raise RuntimeError("Preflight failed; review the saved failed checks")
else:
    print("Technical preflight PASS; scientific status remains PROVISIONAL_RESEARCH.")


## Cell 10 — Fresh Stage 4 Mutual Information

Một fresh worker cho mỗi cấu hình, stdout/stderr và tiến độ checkpoint được lưu ngay lên Drive. Lỗi cấu hình này được báo riêng; báo cáo cấu hình đã xong vẫn giữ nguyên. Resume dùng cùng experiment và chỉ bỏ qua MI đã COMPLETE sau khi verify hash/identity.


In [ ]:
# CELL 10 — Isolated Stage 4 MI, live output and Drive checkpoints
# Reference: stage6-cell-10 guarded fresh worker + streamed output + result JSON.
MI_RESULTS = []
print("=== CICFLOW STAGE 4 — MUTUAL INFORMATION ===")
print("Python:", sys.version.split()[0])
print("Configurations:", SELECTED_CONFIGS)
print("Run IDs:", {config: RUN_IDS[config] for config in SELECTED_CONFIGS})
print("Mode:", MODE)
print("Dataset:", DATASET_ROOT)

# ============================================================
# 1. RECHECK CURRENT PREREQUISITES AND CONFIRMATION
# ============================================================
if MODE in {"full", "resume"}:
    V2.require_launch_context(
        BASE, PREFLIGHT, dataset_ready=DATASET_READY,
        unit_test_evidence=UNIT_TEST_EVIDENCE,
    )
    for SPEC in SPECS:
        SPEC.validate()

    # ========================================================
    # 2. LAUNCH ONE CLEAN PYTHON WORKER PER CONFIGURATION
    # ========================================================
    # run_jobs -> run_worker -> Popen([sys.executable, "-u", ...]).
    # The launcher tees both streams live to per-task and aggregate Drive logs,
    # checks the native exit code and requires one structured worker result.
    # Existing scientific APIs supply MI scores and checkpoint batches.
    print("\nLaunching isolated MI processes...")
    MI_RESULTS = V2.run_jobs("mi", BASE)

    # ========================================================
    # 3. REVIEW EACH WORKER RESULT WITHOUT DISCARDING OTHER RUNS
    # ========================================================
    for result in MI_RESULTS:
        print("\nConfig:", result["config"])
        print("Program status:", result.get("program_status"))
        print("Exit code:", result.get("exit_code"))
        print("Artifact:", result.get("result", {}).get("run_dir"))
        print("Report:", result.get("report_path"))
        if result.get("program_status") != "PASS":
            print("Stopped/failed:", result.get("result", result.get("errors")))
            print("Preserve checkpoints; no automatic restart.")
else:
    MI_RESULTS = [{"program_status": "SKIPPED", "reason": MODE + ": no fit"}]
    print("DRY RUN ONLY — No MI started." if MODE == "dry_run" else "VERIFY MODE — No MI started.")


## Cell 11 — Verify Stage 4 của chính experiment này

Kiểm tra MI score/ranking/order, Top-100, marker digest, Train source identity, tham số và V2 task seal. Không đọc Train feature values. Dry-run chưa fit thì hiển thị NOT_RUN.


In [ ]:
# CELL 11 — Verify MI publication; never fit or repeat an MI computation
# Reference: stage6-cell-11 artifact manifest/hash verification.
MI_VERIFY = []
print("=== STAGE 4 ARTIFACT VERIFICATION ===")
if MODE in {"full", "resume", "verify"}:
    MI_VERIFY = V2.run_jobs("verify_mi", BASE)
    for result in MI_VERIFY:
        artifact = result.get("result", {})
        print("\nConfig:", result["config"], "| verification:", result.get("program_status"))
        print("Run directory:", artifact.get("run_dir"))
        print("Verified artifact hashes:", artifact.get("hashes", {}))
        print("Errors:", artifact.get("errors", []))
        print("Verify report:", result.get("report_path"))
else:
    print("No fitted MI result to verify during dry_run.")
print("No Train/Validation/Test feature values or fitted model were opened by this verification step.")


## Cell 12 — Fresh Stage 5 RF-RFE / SVM branch

Mỗi nhánh có worker/log/manifest riêng. Chỉ chạy với MI mới đã verify ở Cell 11. RFE target=1, step=1 giữ nguyên trong source. Nếu một cấu hình lỗi MI, nhánh của cấu hình đó BLOCKED; các cấu hình đã verify vẫn được xử lý.


In [ ]:
# CELL 12 — Isolated Stage 5 RF-RFE and SVM-branch RF-RFE
# Reference: stage6-cell-10 worker pattern, applied to the independent RFE stage.
RFE_RESULTS = []
print("=== CICFLOW STAGE 5A — RFE ===")
print("Mode:", MODE)
print("Experiments:", [spec.experiment for spec in SPECS])
print("Branches:", SELECTED_BRANCHES)
print("The svm branch's RFE estimator is RandomForestClassifier(max_depth=15).")

# ============================================================
# 1. RECHECK PREREQUISITES FOR THE CURRENT SELECTION
# ============================================================
if MODE in {"full", "resume"}:
    V2.require_launch_context(
        BASE, PREFLIGHT, dataset_ready=DATASET_READY,
        unit_test_evidence=UNIT_TEST_EVIDENCE,
    )
    for SPEC in SPECS:
        SPEC.validate()
    ready = [result["config"] for result in MI_VERIFY if result.get("program_status") == "PASS"]
    blocked = [config for config in SELECTED_CONFIGS if config not in ready]
    if blocked:
        print("RFE BLOCKED: the new MI is not verified for", blocked)
        V2.save_report(RESULT_ROOT, "rfe_upstream_blocked", {"status": "BLOCKED", "configs": blocked})

    # ========================================================
    # 2. LAUNCH DISTINCT WORKERS FOR CONFIGURATION / BRANCH
    # ========================================================
    if ready:
        print("\nLaunching isolated RFE processes...")
        RFE_RESULTS = V2.run_jobs("rfe", {**BASE, "configs": ready})
    for result in RFE_RESULTS:
        print("\nConfig:", result["config"], "| branch:", result["branch"])
        print("Program status:", result.get("program_status"), "| exit:", result.get("exit_code"))
        print("Artifact:", result.get("result", {}).get("run_dir"))
        print("Report:", result.get("report_path"))
        if result.get("program_status") != "PASS":
            print("Stopped/failed:", result.get("result", result.get("errors")))
            print("Preserve this branch's checkpoint; other completed branches remain available.")
else:
    print("DRY RUN ONLY — No RFE started." if MODE == "dry_run" else "VERIFY MODE — No RFE started.")


## Cell 13 — Verify Stage 5

Kiểm tra 100 rank, support, ordered feature mapping với MI, branch identity và seal. Verify không gọi fit hoặc đọc Validation/Test.


In [ ]:
# CELL 13 — Verify RFE publication by branch
# Reference: stage6-cell-11 saved-manifest/hash checks; no refit or Test read.
RFE_VERIFY = []
print("=== STAGE 5A ARTIFACT VERIFICATION ===")
if MODE in {"full", "resume", "verify"}:
    RFE_VERIFY = V2.run_jobs("verify_rfe", BASE)
    for result in RFE_VERIFY:
        artifact = result.get("result", {})
        print("\nConfig:", result["config"], "| branch:", result["branch"])
        print("Verification:", result.get("program_status"))
        print("Run directory:", artifact.get("run_dir"))
        print("Verified artifact hashes:", artifact.get("hashes", {}))
        print("Errors:", artifact.get("errors", []))
        print("Verify report:", result.get("report_path"))
else:
    print("No fitted RFE result to verify during dry_run.")
print("RFE ranking, MI binding and completion seals are checked without another fit.")


## Cell 14 — Tổng hợp cả bốn cấu hình

Theo Cell 12 bản tham chiếu: đọc JSON đã lưu và hiển thị DataFrame nếu pandas có sẵn. Trạng thái verify lấy từ báo cáo worker mới nhất, không từ biến cache trong notebook; marker hiện diện vẫn được hiển thị riêng. Không mở dataset hoặc model cho summary.


In [ ]:
# CELL 14 — Read saved result/report JSON only for summary
# Reference: stage6-cell-12 completed_results() + optional pandas display.
print("=== CICFLOW FOUR-CONFIGURATION RESULTS ===")
SUMMARY = V2.completed_results(RESULT_ROOT)
try:
    import pandas as pd
    display(pd.DataFrame(SUMMARY))
except ImportError:
    for row in SUMMARY:
        print(row)

summary_report = V2.save_report(
    RESULT_ROOT, "summary",
    {"experiment": RUN_INFO["experiment_id"], "rows": SUMMARY,
     "scientific_status": "PROVISIONAL_RESEARCH", "official_reproduction": False},
)
print("Summary report:", summary_report)
print("Verification status is read from saved worker reports, not inferred from a marker or cached notebook variable.")
print("PROVISIONAL_RESEARCH; capture/session independence and CTU-13 reference parity remain unproven.")


## Cell 15 — Lưu inventory và hướng dẫn resume

Kết quả/checkpoint đã được ghi trực tiếp lên Drive theo từng worker và elimination của source; không đợi cell cuối mới backup. Inventory ghi hash các artifact nhỏ, không copy Parquet. Không xóa partial file để ép resume.

Sau khi runtime bị ngắt: Cell 1 đặt MODE=resume và EXISTING_RESULT_ROOT là thư mục V2 đã lưu; chạy lại Cell 2–9, nhập xác nhận ở Cell 9 rồi chạy Cell 10–15. Complete task trong cùng experiment được verify và bỏ qua; checkpoint/final publication không an toàn trả UNSUPPORTED. Thay Python/library/source/dataset sẽ làm identity mismatch và chặn resume.


In [ ]:
# CELL 15 — Preserve artifacts and print exact recovery paths
# Reference: stage6-cell-11 publication paths and stage72-backup small-file policy.
# Results/checkpoints already write directly to Drive during each worker.
INVENTORY = {}
for path in sorted(RESULT_ROOT.rglob("*")):
    if path.is_file() and path.suffix in {".json", ".csv", ".md", ".log"} and "reports" not in path.relative_to(RESULT_ROOT).parts:
        INVENTORY[path.relative_to(RESULT_ROOT).as_posix()] = {
            "bytes": path.stat().st_size, "sha256": V2.file_hash(path),
        }
inventory_path = V2.save_report(
    RESULT_ROOT, "artifact_inventory",
    {"experiment": RUN_INFO["experiment_id"], "artifacts": INVENTORY,
     "scientific_status": "PROVISIONAL_RESEARCH", "official_reproduction": False},
)
print("Drive artifacts:", RESULT_ROOT)
print("Run identity:", RESULT_ROOT / "RUN_INFO.json")
print("Source identity:", RESULT_ROOT / "SOURCE_BINDING.json")
print("Execution identity:", RESULT_ROOT / "EXECUTION_BINDING.json")
print("MI checkpoints:", RESULT_ROOT / "stage4_mi/runs")
print("RFE checkpoints:", RESULT_ROOT / "stage5_rfe/runs")
print("Per-task logs:", RESULT_ROOT / "logs")
print("Aggregate log:", SOURCE_ROOT / "logs/ALL_EXECUTION_RESULTS.log")
print("Reports:", RESULT_ROOT / "reports")
print("Artifact inventory:", inventory_path)
print("\nTo resume this same experiment, use in Cell 1:")
print("MODE = 'resume'")
print("EXISTING_RESULT_ROOT = " + repr(str(RESULT_ROOT)))
print("Then rerun Cells 2–9 and review the exact confirmation before Cells 10–15.")
print("Completed tasks are verified and skipped; unsafe partial publication is UNSUPPORTED.")
print("No Parquet was copied, and no Test evaluation was performed by this publication cell.")
